# Week 8 Tutorial: The Main Ideas of LSTM

This tutorial has **two goals**:

1. help you **understand** the main story of the LSTM lecture;
2. help you **read and debug** the core PyTorch LSTM code.

You do **not** need to build an LSTM cell from scratch.  
You do **not** need to download a dataset.  
You do **not** need to train a large model.

Instead, we focus on the few ideas that connect the lecture directly to practical code.

## The main story in one line

```text
sequence
   ↓
RNN carries information through time using the hidden state h_t
   ↓
long-term information can be difficult to preserve
   ↓
GRU/LSTM introduce gates to control information flow
   ↓
LSTM keeps two recurrent states: hidden state h_t and cell state c_t
   ↓
PyTorch returns output, hn, and cn
   ↓
for many-to-one classification, use the final hidden representation
   ↓
Linear classifier
   ↓
Prediction
```


## What is CORE in this tutorial?

You should be comfortable with these ideas:

1. A review-to-one-label task is **many-to-one**.
2. A simple RNN uses the current input $x_t$ and previous hidden state $h_{t-1}$ to calculate $h_{t}$.
3. Recurrent models reuse the **same parameters at every time step (Parameter Sharing)**.
4. GRU and LSTM use **gates** to control information flow.
5. An LSTM has:
   - hidden state $h_t$: the current representation exposed to the next layer or task;
   - cell state $c_t$: internal memory;
6. With `batch_first=True`, the input is `(B, T, D)`,

   where $B$ = batch size, $T$ = sequence length (number of time steps), $D$ = input feature/embedding dimension.
7. For a one-directional LSTM:
   - `output`: `(B, T, H)`;
   - `hn`: `(L, B, H)`;
   - `cn`: `(L, B, H)`.
   
   where $H$ = hidden size, and $L$ = number of recurrent layers.
8. For many-to-one classification, the classifier receives an $H$-dimensional final hidden representation.
9. `CrossEntropyLoss` uses raw logits and integer class-index targets.
10. You should be able to find common LSTM code errors by checking shapes and meanings.


## Setup

Only PyTorch is needed.

This notebook is small enough to run on CPU. A GPU is **not required**.


In [7]:
import torch
import torch.nn as nn

torch.manual_seed(0)

print("PyTorch version:", torch.__version__)


PyTorch version: 2.11.0+cpu


# Part 1: from RNN to LSTM, the Key Concepts


## 1.1 Sequence modelling and many-to-one classification

A sentence is a sequence:

$$
x_1,x_2,\ldots,x_T.
$$

Suppose a model reads an entire movie review and predicts **one** sentiment label:

```text
word 1 → word 2 → word 3 → ... → word T
                                   ↓
                           one final label
```

Many input time steps produce one output.

This is called a **many-to-one** sequence task.

A sentiment classifier is a standard example.

## 1.2 What does a simple RNN do?

At time step $t$, a simple RNN updates its hidden state using

$$
x_t
\quad\text{and}\quad
h_{t-1}.
$$

A simplified view is

$$
h_t=f(x_t,h_{t-1}).
$$

Two ideas matter here.

### Idea 1: information moves through time

The hidden state carries information from earlier positions toward later positions.

### Idea 2: parameter sharing across time

The **same recurrent parameters** are reused at every time step.

The RNN does **not** learn one completely different set of recurrent weights for time step 1, another set for time step 2, and so on.


## 1.3 Why introduce GRU and LSTM?

A simple RNN can struggle when useful information must be carried across many time steps.

One important reason is the **vanishing-gradient problem** during backpropagation through time: information from far earlier positions can become difficult to learn and preserve.

GRU and LSTM introduce **gates**.

A gate is an information-flow controller.

It helps the model decide:

- what old information to keep;
- what information to update;
- what information to expose as the current output.

The important idea is **information flow**, not memorising every gate equation.


## 1.4 GRU versus LSTM

The most important state difference is simple.

| Model | Recurrent state |
|---|---|
| Simple RNN | one hidden state $h_t$ |
| GRU | one **gated** hidden state $h_t$ |
| LSTM | hidden state $h_t$ **and** separate cell state $c_t$ |

For an LSTM:

- $c_t$ is the **cell state**: think of it as internal memory;
- $h_t$ is the **hidden state**: think of it as the current representation exposed to the next layer or task.

A useful analogy is:

> **Cell state $c_t$ = notebook memory**  
> **Hidden state $h_t$ = what the model reads from the notebook now**


## 1.5 What do the three LSTM gates mean?

You do **not** need to implement these gates manually in this tutorial.

You only need their roles.

| Gate | Main role |
|---|---|
| Forget gate $f_t$ | controls how much old cell memory $c_{t-1}$ is kept |
| Input gate $i_t$ | controls how much new candidate information $\tilde{c}_t$ is written |
| Output gate $o_t$ | controls how much updated cell memory is exposed as $h_t$ |

So the lecture's main LSTM idea can be summarised as:

```text
old memory, c_t-1             new candidate information, ~c_t
   ↓                                       ↓
[forget gate] forget / keep       [Input gate] write
   ↓                                       ↓
   ---------------->   +   <----------------
                       ↓
                       ↓
              updated cell state c_t
                       ↓
                  [Output gate] expose
                       ↓
                 hidden state h_t
```


### Optional: the equations from the lecture

You do not need to memorise these equations for this tutorial.

<details>
<summary><b>Show the standard LSTM equations</b></summary>

$$
f_t=\sigma\left(W_f[h_{t-1},x_t]+b_f\right)
$$

$$
i_t=\sigma\left(W_i[h_{t-1},x_t]+b_i\right)
$$

$$
o_t=\sigma\left(W_o[h_{t-1},x_t]+b_o\right)
$$

$$
\tilde{c}_t=\tanh\left(W_c[h_{t-1},x_t]+b_c\right)
$$

$$
c_t=f_t\odot c_{t-1}+i_t\odot\tilde{c}_t
$$

$$
h_t=o_t\odot\tanh(c_t)
$$

The important interpretation is still:

- forget gate → keep old memory;
- input gate → write new information;
- output gate → expose part of the updated memory.

</details>


# Part 2: The PyTorch view  `nn.LSTM`

This is the most important practical part of the tutorial.

We will use PyTorch's built-in LSTM:

```python
nn.LSTM(...)
```

**We treat the internal gates as already implemented for us.**

Our job is to understand:⚠️⚠️⚠️

1. what shape goes **into** the layer❓
2. what tensors come **out**❓
3. which output is useful for a many-to-one classifier❓


## 2.1 The shape symbols

We use:

- $B$ = batch size;
- $T$ = sequence length / number of time steps;
- $D$ = input size at each time step;
- $H$ = hidden size;
- $L$ = number of recurrent layers.

With

```python
batch_first=True
```

the input shape is

$$
(B,T,D).
$$

For a one-directional LSTM:

$$
\texttt{output}:(B,T,H)
$$

$$
\texttt{hn}:(L,B,H)
$$

$$
\texttt{cn}:(L,B,H).
$$

Important, `batch_first=True` changes the layout of the **input and output sequence tensors**.

It does **not** change `hn` and `cn` into `(B,L,H)`.

They remain `(L,B,H)`.


## 2.2 Run one tiny LSTM

We use deliberately small dimensions:

```text
B = 4 samples
T = 7 time steps
D = 10 input features
H = 16 hidden features
L = 2 LSTM layers
```

Before running the code, predict:

```text
input x: ?
output : ?
hn     : ?
cn     : ?
```


In [8]:
# ------------------------------------------------------------
# A small two-layer LSTM
# ------------------------------------------------------------
B = 4   # batch size
T = 7   # sequence length
D = 10  # input size
H = 16  # hidden size
L = 2   # number of LSTM layers

lstm = nn.LSTM(
    input_size=D,
    hidden_size=H,
    num_layers=L,
    batch_first=True
)

# batch_first=True means:
# x.shape = (batch size, sequence length, input size)
# torch.randn: generates random numbers from a standard normal distribution.
#              Here, we are using it only to simulate a batch of sequence data.
x = torch.randn(B, T, D)

# An LSTM returns TWO things:
#   1. output
#   2. a tuple containing (hn, cn)
output, (hn, cn) = lstm(x)

print("x shape:     ", x.shape)
print("output shape:", output.shape)
print("hn shape:    ", hn.shape)
print("cn shape:    ", cn.shape)


x shape:      torch.Size([4, 7, 10])
output shape: torch.Size([4, 7, 16])
hn shape:     torch.Size([2, 4, 16])
cn shape:     torch.Size([2, 4, 16])


### How to read the result ⚠️

For the example above:

```text
x      : (4, 7, 10)
output : (4, 7, 16)
hn     : (2, 4, 16)
cn     : (2, 4, 16)
```

#### `output`
contains the hidden representation at **every time step** from the final LSTM layer:

$$
(B,T,H).
$$

So each of the 4 samples has 7 hidden vectors, and each hidden vector has length 16.

#### `hn`
contains the **final hidden state from each LSTM layer**:

$$
(L,B,H).
$$

There are 2 layers, so the first dimension is 2.

#### `cn`
contains the **final cell state from each LSTM layer**:

$$
(L,B,H).
$$

It follows the same shape pattern as `hn`.


## 2.3 One shape rule worth remembering

For a one-directional recurrent layer with `batch_first=True`:

```text
input  : (B, T, D)

output : (B, T, H)

hn     : (L, B, H)

LSTM only:
cn     : (L, B, H)
```

The same `output` / `hn` pattern is also useful when reading ordinary RNN and GRU code.⚠️

LSTM adds the separate `cn`.


### Practice: Shape checkpoint

An LSTM has:

```text
B = 5
T = 6
D = 12
H = 20
L = 3
```

and `batch_first=True`.

What are the shapes of:

1. `x`
2. `output`
3. `hn`
4. `cn`

<details>
<summary><b>Check answer</b></summary>

```text
x      : (5, 6, 12)
output : (5, 6, 20)
hn     : (3, 5, 20)
cn     : (3, 5, 20)
```

</details>


# Part 3: Many-to-one LSTM classification



Suppose the whole sequence should produce **one class label**.

A simple architecture is:

```text
sequence x
   ↓
LSTM
   ↓
final hidden representation in the final layer (important)
   ↓
Linear layer
   ↓
class logits
```

The final hidden representation has size $H$.

Therefore, for $C$ classes, the classifier should be

```python
nn.Linear(hidden_size, num_classes)
```

not

```python
nn.Linear(input_size, num_classes)
```

Because the LSTM has already transformed the original $D$-dimensional input features into an $H$-dimensional hidden representation.

## 3.1 Which tensor represents the final sequence?

With `batch_first=True`:

```python
output.shape == (B, T, H)
```

To select the final time step for **every sample**:

```python
output[:, -1, :]
```

Its shape is

```text
(B, H)
```

Notice the difference:

```python
output[-1]
```

selects the final **batch item**, not the final time step.

For a one-directional LSTM, another convenient choice is the final hidden state from the top recurrent layer:

```python
hn[-1]
```

which also has shape `(B,H)`.


In [9]:
# Compare the two final-hidden representations.✅
last_from_output = output[:, -1, :]  # final time step for every sample
last_from_hn = hn[-1]                # final hidden state from top LSTM layer

print("output[:, -1, :]:", last_from_output.shape)
print("hn[-1]:           ", last_from_hn.shape)

# For this one-directional LSTM they represent the same final hidden state.
print(
    "Same values:",
    torch.allclose(last_from_output, last_from_hn)
)


output[:, -1, :]: torch.Size([4, 16])
hn[-1]:            torch.Size([4, 16])
Same values: True


## 3.2 A complete minimal classifier

The following model is intentionally short.

Read the comments carefully.

You do **not** need to memorise the class syntax.  
You should understand the shape and meaning of each important line.


In [10]:
class SimpleLSTMClassifier(nn.Module):
    # nn.Module: Inherit from PyTorch's base neural network class nn.Module
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        # `super().__init__()` initialises the parent `nn.Module` class
        # so that PyTorch can correctly manage the layers and trainable
        # parameters in our model.

        # ----- Define the LSTM + Linear Layer -----
        # The LSTM receives one input_size vector at each time step.
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        # The final hidden representation has size hidden_size (= H),
        # so the classifier must receive H features.
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        '''
        we define the model architecture in `__init__()`,
        while `forward()` defines how the data flows through the model.

        input x: (B, T, D)
        '''

        # LSTM return structure:
        # output: (B, T, H)
        # hn:     (1, B, H) Becasue this is a one-layer LSTM
        # cn:     (1, B, H) Becasue this is a one-layer LSTM
        output, (hn, cn) = self.lstm(x)

        # Select the final time step for every sample.
        # Shape: (B, H)
        last_hidden = output[:, -1, :]

        # Convert the H-dimensional sequence representation
        # into C raw class scores.
        # Shape: (B, C)
        logits = self.fc(last_hidden)

        return logits


In [11]:
# ------------------------------------------------------------
# A tiny shape test -- no training required
# ------------------------------------------------------------
input_size = 10
hidden_size = 32
num_classes = 4

model = SimpleLSTMClassifier(
    input_size=input_size,
    hidden_size=hidden_size,
    num_classes=num_classes
)

# B=8, T=5, D=10
x = torch.randn(8, 5, input_size)

logits = model(x)

print("input shape: ", x.shape)
print("logits shape:", logits.shape)


input shape:  torch.Size([8, 5, 10])
logits shape: torch.Size([8, 4])


## 3.3 Logits, Labels, and `CrossEntropyLoss`


### 3.3.1 Logits
For classification, our model returns **logits**.

If the batch size is $B$ and the number of classes is $C$, then:

```text
logits.shape = (B, C)
labels.shape = (B,)
```

For each sample in the batch, the model produces one logit for each class.

For example, if we have four classes, one sample may produce:

```text
logits = [2.1, -0.4, 1.3, 0.8]
```

These values are **raw model outputs**. They are not probabilities yet, so they do not need to add up to 1.

Conceptually:

```text
raw logits
(B, C)
   ↓
softmax (inside CrossEntropyLoss in pytorch)
   ↓
probabilities
(B, C)
```

For example:

```text
logits for one sample:
[2.1, -0.4, 1.3, 0.8]

        ↓ softmax

probabilities:
[0.56, 0.05, 0.25, 0.14]
```

The shape does not change. Softmax only converts the raw logits into probabilities.

**Do We Need to Add Softmax Ourselves?** **No.**

When we use:

```python
criterion = nn.CrossEntropyLoss()
```

we should pass the **raw logits directly** to `CrossEntropyLoss`.

We do **not** need to add a `Softmax` layer before it because `CrossEntropyLoss` already performs the required softmax-related calculation internally.

So the actual PyTorch workflow is:

```text
model
  ↓
raw logits
(B, C)
  ↓
CrossEntropyLoss
  ↑
integer class labels
(B,)
```


### 3.3.2 Labels

The labels are stored as **integer class indices**.

For example, if we have four classes:

```text
class 0 → label 0
class 1 → label 1
class 2 → label 2
class 3 → label 3
```

A batch of labels may look like:

```text
labels = [2, 0, 3, 1, ...]
```

Therefore:

```text
labels.shape = (B,)
```

and the target dtype should be:

```python
torch.long
```

We do not need to convert these labels into probabilities or one-hot vectors for the standard use of `CrossEntropyLoss`.


### 3.3.3 Cross-Entropy in Our Two-Class Sentiment Example

In our sentiment classification task, we have two classes:

```text
0 → negative
1 → positive
```

For one sample, the model produces two logits:

$$
z_0,\; z_1
$$

Softmax converts them into two probabilities:

$$
p_0 = \frac{e^{z_0}}{e^{z_0} + e^{z_1}}
$$

$$
p_1 = \frac{e^{z_1}}{e^{z_0} + e^{z_1}}
$$

For our two-class sentiment classification problem, the cross-entropy loss can be written as:

$$
L = -\left[y_0\log(p_0) + y_1\log(p_1)\right]
$$

where $p_0$ and $p_1$ are the predicted probabilities for the negative and positive classes, and $y_0$ and $y_1$ indicate the correct class.

For example, if the correct class is positive, then $y_0=0$ and $y_1=1$, so:

$$
L = -\log(p_1)
$$

This means the loss becomes smaller when the model gives a higher probability to the correct class.

So conceptually:

```text
raw logits
[z0, z1]
   ↓
softmax
   ↓
[p0, p1]
   ↓
select the probability of the correct class
   ↓
-log(correct-class probability)
   ↓
cross-entropy loss
```

⚠️ **Important:** `nn.CrossEntropyLoss()` expects **raw logits** and **integer class labels**. Do not apply `Softmax` to the logits before passing them to `CrossEntropyLoss`.

In [12]:
# `torch.randint()` generates random integers.
# Here, we use it to simulate a batch of 8 class labels.
labels = torch.randint(
    low=0,               # Smallest possible integer label, inclusive
    high=num_classes,    # Largest possible label is num_classes - 1
    size=(8,),           # Generate 8 labels, so labels.shape = (8,)
    dtype=torch.long     # Integer type required by nn.CrossEntropyLoss
)

criterion = nn.CrossEntropyLoss()

loss = criterion(logits, labels)

print("labels shape:", labels.shape)
print("labels dtype:", labels.dtype)
print("loss:", float(loss))


labels shape: torch.Size([8])
labels dtype: torch.int64
loss: 1.4185903072357178


# Part 4: Practice with RNN and LSTM



The purpose of this part is
- to help you transfer what you have learned about LSTM to a closely related recurrent model, the simple RNN.
- to help you preapre the final exam

The PyTorch usage of an RNN is very similar to the usage of an LSTM.

For both models, with `batch_first=True`:

```text
input  : (B, T, D)
output : (B, T, H)
hn     : (L, B, H)
```

The main return value difference is simple:

```python
# RNN
output, hn = rnn(x)

# LSTM
output, (hn, cn) = lstm(x)
```

An RNN does not have the separate cell state `cn`.

For a many to one classification task, the overall idea is also the same:

```text
sequence
   ↓
RNN or LSTM
   ↓
final hidden representation
   ↓
Linear classifier
   ↓
class logits
```

So if you understand the LSTM code from Part 3, the following RNN code should look very familiar.


## 4.1 Practice: RNN Code Correction

The following program is intended to perform a three class, many to one sequence classification task.

Assume:

```text
B = 6
T = 7
D = 12
H = 20
C = 3
```

The intended design is:

```text
input x                 : (B, T, D)
RNN output              : (B, T, H)
final sequence feature  : (B, H)
logits                  : (B, C)
labels                  : (B,)
```

The model uses `CrossEntropyLoss`.

The code contains **exactly five errors**.

Your task is to identify the five incorrect lines and give a correct replacement for each one.

```python
import torch
import torch.nn as nn

class RNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=False
        )

        self.fc = nn.Linear(hidden_size, num_classes + 1)

    def forward(self, x):
        output, hn = self.rnn(x)

        last_hidden = output[:, 0, :]

        logits = self.fc(last_hidden)
        probabilities = torch.softmax(logits, dim=1)

        return probabilities


input_size = 12
hidden_size = 20
num_classes = 3

model = RNNClassifier(
    input_size=input_size,
    hidden_size=hidden_size,
    num_classes=num_classes
)

x = torch.randn(6, 7, input_size)

labels = torch.randint(
    0,
    num_classes,
    (6, 1),
    dtype=torch.long
)

criterion = nn.CrossEntropyLoss()

logits = model(x)
loss = criterion(logits, labels)
```

Try to find all five errors before opening the answer.


### Answer and Explanation

<details>
<summary><b>Show the five corrections</b></summary>

### Correction 1: `batch_first`

Incorrect:

```python
batch_first=False
```

Correct:

```python
batch_first=True
```

The input has shape `(B, T, D)`, so `batch_first=True` is required.

### Correction 2: Number of classifier outputs

Incorrect:

```python
self.fc = nn.Linear(hidden_size, num_classes + 1)
```

Correct:

```python
self.fc = nn.Linear(hidden_size, num_classes)
```

The classifier should produce one logit for each class.

### Correction 3: Select the final time step

Incorrect:

```python
last_hidden = output[:, 0, :]
```

Correct:

```python
last_hidden = output[:, -1, :]
```

With `output` shaped `(B, T, H)`, `[:, -1, :]` selects the final time step for every sample.

For a one directional RNN, `hn[-1]` is also acceptable.

### Correction 4: Return raw logits

Incorrect:

```python
probabilities = torch.softmax(logits, dim=1)
return probabilities
```

Correct:

```python
return logits
```

`nn.CrossEntropyLoss()` expects raw logits, so an extra softmax is not needed.

### Correction 5: Label shape

Incorrect:

```python
labels = torch.randint(
    0,
    num_classes,
    (6, 1),
    dtype=torch.long
)
```

Correct:

```python
labels = torch.randint(
    0,
    num_classes,
    (6,),
    dtype=torch.long
)
```

For class index targets:

```text
logits : (B, C)
labels : (B,)
```

The dtype `torch.long` is already correct.

</details>

## 4.2 Multiple Choice Practice

### Question 1

An RNN uses `batch_first=True`, and its `output` tensor has shape:

```text
(B, T, H)
```

What does the following line select?

```python
final_feature = output[:, -1, :]
```

A. The final sample in the batch

B. The final time step for every sample

C. The final hidden feature only

D. All time steps from the first sample

<details>
<summary><b>Answer and explanation</b></summary>

**Answer: B**

`output` has three dimensions:

```text
batch, time, hidden feature
```

The expression:

```python
output[:, -1, :]
```

means:

```text
all samples
final time step
all hidden features
```

Therefore the result has shape:

```text
(B, H)
```

This is useful for many to one classification because it gives one final sequence representation for each sample.

</details>


### Question 2

Consider:

```python
rnn = nn.RNN(
    input_size=7,
    hidden_size=18,
    num_layers=3,
    batch_first=True
)

x = torch.randn(5, 9, 7)
output, hn = rnn(x)
```

What is the shape of `hn`?

A. `(5, 9, 18)`

B. `(3, 5, 18)`

C. `(5, 3, 18)`

D. `(3, 9, 18)`

<details>
<summary><b>Answer and explanation</b></summary>

**Answer: B**

For a one directional RNN:

```text
hn : (L, B, H)
```

Here:

```text
L = 3
B = 5
H = 18
```

so:

```text
hn : (3, 5, 18)
```

Notice that `batch_first=True` changes the layout of the input and `output`.

It does not change the layout of `hn`.

</details>


### Question 3

Suppose a classifier produces:

```text
logits : (8, 4)
```

and uses:

```python
criterion = nn.CrossEntropyLoss()
```

Which target tensor is most appropriate?

A. Shape `(8, 4)`, dtype `torch.float32`

B. Shape `(8,)`, dtype `torch.long`

C. Shape `(4,)`, dtype `torch.long`

D. Shape `(8, 1)`, dtype `torch.float32`

<details>
<summary><b>Answer and explanation</b></summary>

**Answer: B**

There are 8 samples and each sample has one class index.

Therefore the target shape is:

```text
(8,)
```

For class index targets, `CrossEntropyLoss` expects integer labels, normally:

```python
torch.long
```

The model should provide raw logits with shape `(B, C)`.

A separate softmax is not required before the loss.

</details>




### Question 4

A simple RNN uses:

$$
h_t=\tanh(W_{xh}x_t+W_{hh}h_{t-1}+b_h).
$$

Suppose:

$$
h_{t-1}\in\mathbb{R}^{20}
$$

What must be the shape of $W_{hh}$?

A. $20\times20$

B. $20\times1$

C. $1\times20$

D. $40\times20$

<details>
<summary><b>Answer and explanation</b></summary>

**Answer: A**

The multiplication:

$$
W_{hh}h_{t-1}
$$

must produce a 20 dimensional vector so that it can contribute to the new hidden state $h_t$.

Therefore:

$$
(20\times20)(20\times1)
\rightarrow
(20\times1).
$$

So:

$$
W_{hh}\in\mathbb{R}^{20\times20}.
$$

The same matrix multiplication rule applies throughout neural network shape reasoning.

</details>
